# Filter Metadata file 
Drop all Acne_NL samples that have a score bigger than 0

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

import biom
from biom import load_table
from qiime2 import Artifact
import qiime2 as q2

In [3]:
# Read the metadata file
metadata_file = '../Metadata/metadata_final_22102024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

# Now filter out the samples that match 'low Acne_NL'
samples_to_drop = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']

# Get the number of samples that are being dropped
num_dropped_samples = len(samples_to_drop)

# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Drop the samples
metadata_df = metadata_df[metadata_df['severity_group'] != 'low Acne_NL']

# Output the number of dropped samples
print(f'Number of samples dropped: {num_dropped_samples}')

Number of samples dropped: 23


In [4]:
# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

In [5]:
# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

In [ ]:
# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
        # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")

    # Transpose the filtered biom table
    biom_tbl_filtered_T = biom_tbl_filtered.transpose()

    # Convert the transposed BIOM table to a QIIME 2 Artifact
    biom_tbl_filtered_artifact = q2.Artifact.import_data('FeatureTable[Frequency]', biom_tbl_filtered_T)
    
    # Save the artifact as a .qza file
    biom_tbl_filtered_artifact.save(f'../Data/16S/CTF/filtered_{dataset_id}.qza')

In [ ]:
# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/CTF/174951_rarefied_table_unannotated_absolute_abundances_T.qza'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances_T.qza')
]